### Data Profiling

In [2]:
import pandas as pd
from ydata_profiling import ProfileReport

C:\Users\MAGICO\AppData\Local\Temp\ipykernel_2624\2829373970.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [3]:
Sales_Pipeline = pd.read_csv('Raw Data/sales_pipeline.csv')
Accounts = pd.read_csv('Raw Data/accounts.csv')
Products = pd.read_csv('Raw Data/products.csv')
Sales_teams = pd.read_csv('Raw Data/sales_teams.csv')

In [4]:
print("Sales_Pipeline shape:", Sales_Pipeline.shape)
print("Accounts shape:", Accounts.shape)
print("Products shape:", Products.shape)
print("Sales_team shape:", Sales_teams.shape)

Sales_Pipeline shape: (8800, 8)
Accounts shape: (85, 7)
Products shape: (7, 3)
Sales_team shape: (35, 3)


In [5]:
Profile_Sales_Pipeline = ProfileReport(Sales_Pipeline, title="Sales Pipeline Profiling Report", explorative=True)

In [6]:
Profile_Sales_Pipeline. to_file("Sales_Pipeline_Profiling_Report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00,  9.54it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
Sales_Pipeline.groupby("deal_stage")["engage_date"].apply(lambda x: x.isna().sum())    

deal_stage
Engaging         0
Lost             0
Prospecting    500
Won              0
Name: engage_date, dtype: int64

In [9]:
Sales_Pipeline.groupby("deal_stage")["close_date"].apply(lambda x: x.isna().sum())

deal_stage
Engaging       1589
Lost              0
Prospecting     500
Won               0
Name: close_date, dtype: int64

In [10]:
Sales_Pipeline.groupby("deal_stage")["close_value"].apply(lambda x: x.isna().sum())

deal_stage
Engaging       1589
Lost              0
Prospecting     500
Won               0
Name: close_value, dtype: int64

In [13]:
Profile_Accounts = ProfileReport(Accounts, title="Accounts Profiling Report", explorative=True)
Profile_Accounts.to_file("Accounts_Profiling_Report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 479.22it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
Profile_Products = ProfileReport(Products, title="Products Profiling Report", explorative=True)
Profile_Products.to_file("Products_Profiling_Report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 78.75it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
Profile_Sales_teams = ProfileReport(Sales_teams, title="Sales Teams Profiling Report", explorative=True)
Profile_Sales_teams.to_file("Sales_Teams_Profiling_Report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 16.94it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### Relationship Validation

In [18]:
# Sales Agents
Sales_Pipeline[Sales_Pipeline["sales_agent"].notna() 
               & ~Sales_Pipeline["sales_agent"].isin(Sales_teams["sales_agent"])]["sales_agent"].unique()

array([], dtype=object)

In [19]:
# Products
Sales_Pipeline[Sales_Pipeline["product"].notna() 
               & ~Sales_Pipeline["product"].isin(Products["product"])]["product"].unique()

array(['GTXPro'], dtype=object)

In [21]:
Products["product"].unique()

array(['GTX Basic', 'GTX Pro', 'MG Special', 'MG Advanced',
       'GTX Plus Pro', 'GTX Plus Basic', 'GTK 500'], dtype=object)

In [23]:
Sales_Pipeline["product"].unique()

array(['GTX Plus Basic', 'GTXPro', 'MG Special', 'GTX Basic',
       'MG Advanced', 'GTX Plus Pro', 'GTK 500'], dtype=object)

In [20]:
# Accounts
Sales_Pipeline[Sales_Pipeline["account"].notna() 
               & ~Sales_Pipeline["account"].isin(Accounts["account"])]["account"].unique()

array([], dtype=object)

## Data Quality Summary

The dataset was reviewed across the four main tables: `accounts`, `products`, `sales_teams`, and `sales_pipeline`.

The profiling and validation checks focused on:

* Missing values
* Duplicate records
* Uniqueness of key candidate columns
* Relationships between tables
* Business logic consistency
* Naming inconsistencies

The raw CSV files were kept unchanged. Any required cleaning or standardization will be performed during the ETL/transformation stage.


### 1. Product Name Inconsistency

A naming inconsistency was identified between the `products` and `sales_pipeline` tables.

* In `products`, the product is recorded as `GTX Pro`.
* In `sales_pipeline`, the same product is recorded as `GTXPro`.

This inconsistency causes `GTXPro` to appear as an unmatched product during relationship validation.

**Action:** The raw data will not be modified. The product name will be standardized during the ETL/transformation stage.


### 2. Sales Pipeline — Missing Values

Missing values were identified in `account`, `engage_date`, `close_date`, and `close_value`.

The missing values in `engage_date`, `close_date`, and `close_value` are related to the current `deal_stage`:

* `Prospecting` opportunities do not have an `engage_date`.
* `Engaging` opportunities do not have a `close_date` or `close_value`.
* `Won` and `Lost` opportunities have `close_date` and `close_value`.

These missing values are considered consistent with the sales process rather than automatically treated as data errors.

No rows were removed because of these missing values.


### 3. Close Value Business Rule

The `close_value` column was checked against `deal_stage`.

All `Lost` opportunities have a `close_value` of `0`, while opportunities that are still in progress may have a missing `close_value`.

This behavior is consistent with the business meaning of the sales stages and was therefore not treated as an error.


### 4. Accounts — Subsidiary Relationship

The `subsidiary_of` column contains missing values for most accounts.

This is considered a valid business condition because not every account is necessarily a subsidiary of another company.

The column also indicates a self-referencing relationship within the `accounts` table, where an account can reference another account as its parent company.

No missing values were filled during profiling.


### 5. Accounts — Naming Inconsistencies

Some spelling inconsistencies were observed in categorical values, such as:

* `technolgy`
* `Philipines`

These values were not modified in the raw data.

They should be reviewed and standardized during the transformation stage if they affect reporting or dimensional consistency.
